# 04 — Train Encoders

Pretrain all three encoders independently:
- **EEG**: seizure detection (LOO-CV, 50 epochs)
- **MRI**: normal vs delayed myelination (30 epochs)
- **HPO**: binary ID classification (100 epochs)

Saves checkpoints to `checkpoints/`. Requires T4 GPU.

**Run order: 01 → 02 → 03 → 04 → 05 → 06**

In [ ]:
# CELL 1: Clone / pull repo
import os
if not os.path.exists('/content/earlyMind'):
    !git clone https://github.com/Rickykapoor/earlyMind.git /content/earlyMind
%cd /content/earlyMind
!git pull origin main

In [ ]:
# CELL 2: Install dependencies
!pip install -q mne nibabel nilearn openneuro-py torch torchvision \
  streamlit plotly scikit-learn pytorch-tabnet \
  xgboost catboost tqdm pyyaml scipy

In [ ]:
# CELL 3: Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 4: Download processed datasets from GitHub Releases
# NOTE: Run notebooks 01, 02, 03 first to generate processed datasets.
# If re-running 04 standalone, download pre-processed data:
import os

RELEASE = 'https://github.com/Rickykapoor/earlyMind/releases/download/v1.0.0-data'

# Download raw archives (needed if processed dirs are empty)
if not os.path.exists('datasets/processed/eeg'):
    !mkdir -p datasets/eeg/helsinki_neonatal
    !wget -qO eeg_raw.tar.gz {RELEASE}/eeg_raw.tar.gz
    !tar -xzf eeg_raw.tar.gz -C datasets/eeg

if not os.path.exists('datasets/processed/mri'):
    !mkdir -p datasets/mri/baby_open_brains
    !wget -qO mri_raw.tar.gz {RELEASE}/mri_raw.tar.gz
    !tar -xzf mri_raw.tar.gz -C datasets/mri

if not os.path.exists('datasets/processed/facial'):
    !mkdir -p datasets/facial/hpo
    !wget -qO facial_raw.tar.gz {RELEASE}/facial_raw.tar.gz
    !tar -xzf facial_raw.tar.gz -C datasets/facial

print('Datasets ready.')

In [ ]:
# CELL 5: Setup paths
import sys
sys.path.insert(0, '/content/earlyMind')

import torch
import numpy as np
from pathlib import Path
from src.config import cfg

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg.paths.makedirs()
print('Config loaded. Checkpoints dir:', cfg.paths.checkpoints)

In [ ]:
# CELL 6: PART 1 — Pretrain EEG Encoder (seizure detection, LOO-CV)
from src.models.eeg_encoder import EEGEncoder, EEGSeizureDataset
from src.training.trainer import pretrain_encoder
import torch.nn as nn
from torch.utils.data import DataLoader
from src.data.eeg_loader import process_eeg_dataset

eeg_proc = cfg.paths.eeg_processed
epoch_files = sorted(eeg_proc.glob('*_epochs.npy'))

import pandas as pd
label_df = pd.read_csv(eeg_proc / 'labels.csv') if (eeg_proc / 'labels.csv').exists() else None

records = []
for ep in epoch_files:
    sid = ep.stem.replace('_epochs', '')
    lbl, dq = 0, 90.0
    if label_df is not None:
        row = label_df[label_df['subject_id'].astype(str) == sid]
        if len(row) > 0:
            lbl = int(row.iloc[0]['label'])
            dq  = float(row.iloc[0]['dq'])
    records.append({'subject_id': sid, 'epoch_path': str(ep), 'label': lbl, 'dq': dq})

print(f'EEG subjects: {len(records)}')

# LOO-CV pretraining
for val_idx in range(len(records)):
    train_recs = [r for i, r in enumerate(records) if i != val_idx]
    val_recs   = [records[val_idx]]

    train_ds = EEGSeizureDataset(train_recs, augment=True)
    val_ds   = EEGSeizureDataset(val_recs,   augment=False)
    train_ldr = DataLoader(train_ds, batch_size=4, shuffle=True)
    val_ldr   = DataLoader(val_ds,   batch_size=4)

    enc = EEGEncoder(embed_dim=128)
    loss_fn = nn.BCEWithLogitsLoss()

    history = pretrain_encoder(
        encoder=enc,
        train_loader=train_ldr,
        val_loader=val_ldr,
        n_epochs=50,
        lr=1e-3,
        device=device,
        save_path=str(cfg.paths.checkpoints / f'eeg_loo_{val_idx}.pt'),
        loss_fn=loss_fn,
        get_logits_fn=lambda m, x: m(x.to(device), pretrain=True)['seizure_logit'],
        task_name=f'EEG LOO fold {val_idx}',
        patience=10,
    )

torch.save(enc.state_dict(), str(cfg.paths.checkpoints / 'eeg_encoder_pretrained.pt'))
print('EEG encoder saved.')

In [ ]:
# CELL 7: PART 2 — Pretrain MRI Encoder (myelination status)
from src.models.mri_encoder import MRIEncoder, MRIPretrainDataset
from sklearn.model_selection import train_test_split

mri_files = sorted(cfg.paths.mri_processed.glob('sub-*.npy'))
mri_records = [{'out_path': str(f), 'label': 0} for f in mri_files]
print(f'MRI subjects: {len(mri_records)}')

train_recs, val_recs = train_test_split(mri_records, test_size=0.2, random_state=42)
train_ds = MRIPretrainDataset(train_recs, augment=True)
val_ds   = MRIPretrainDataset(val_recs,   augment=False)
train_ldr = DataLoader(train_ds, batch_size=4, shuffle=True)
val_ldr   = DataLoader(val_ds,   batch_size=4)

mri_enc = MRIEncoder(embed_dim=128)
mri_loss = nn.BCEWithLogitsLoss()

mri_history = pretrain_encoder(
    encoder=mri_enc,
    train_loader=train_ldr,
    val_loader=val_ldr,
    n_epochs=30,
    lr=5e-4,
    device=device,
    save_path=str(cfg.paths.checkpoints / 'mri_encoder_pretrained.pt'),
    loss_fn=mri_loss,
    get_logits_fn=lambda m, x: m(x.to(device), pretrain=True)['myelin_logit'],
    task_name='MRI Myelination',
    patience=8,
)
print('MRI encoder saved.')

In [ ]:
# CELL 8: PART 3 — Pretrain HPO Encoder (binary ID classification)
from src.models.hpo_encoder import HPOEncoder, HPOPretrainDataset

X = np.load(str(cfg.paths.hpo_processed / 'hpo_matrix.npy'))
y = np.load(str(cfg.paths.hpo_processed / 'hpo_labels.npy'))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

train_ds = HPOPretrainDataset(X_tr, y_tr)
val_ds   = HPOPretrainDataset(X_te, y_te)
train_ldr = DataLoader(train_ds, batch_size=32, shuffle=True)
val_ldr   = DataLoader(val_ds,   batch_size=32)

pos_frac   = y_tr.mean()
pos_weight = torch.tensor([(1 - pos_frac) / (pos_frac + 1e-8)], dtype=torch.float32)
print(f'HPO pos_weight: {pos_weight.item():.2f}')

n_hpo   = X.shape[1]
hpo_enc = HPOEncoder(n_hpo=n_hpo, embed_dim=128)
hpo_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

hpo_history = pretrain_encoder(
    encoder=hpo_enc,
    train_loader=train_ldr,
    val_loader=val_ldr,
    n_epochs=100,
    lr=1e-3,
    device=device,
    save_path=str(cfg.paths.checkpoints / 'hpo_encoder_pretrained.pt'),
    loss_fn=hpo_loss,
    get_logits_fn=lambda m, x: m(x.to(device), pretrain=True)['id_logit'],
    task_name='HPO ID classifier',
    patience=15,
)
print('HPO encoder saved.')

In [ ]:
# CELL 9: Plot training histories
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, hist, name in zip(axes,
    [history, mri_history, hpo_history],
    ['EEG', 'MRI', 'HPO']):
    ax.plot(hist['train_loss'], label='train')
    ax.plot(hist['val_loss'],   label='val')
    ax.set_title(f'{name} Encoder Loss')
    ax.set_xlabel('Epoch')
    ax.legend()
plt.tight_layout()
plt.savefig(str(cfg.paths.reports) + '/encoder_train_histories.png', dpi=100)
plt.show()

In [ ]:
# CELL 10: Commit and push results
!git add checkpoints/ reports/ datasets/processed/
!git commit -m 'colab: 04_train_encoders completed'
!git push origin main